# Project 1 - Slideflow MSI Pipeline On TCGA-CRC WSIs

Weakly supervised MSI-H vs MSS classifier using TCGA-CRC diagnostic WSIs, Slideflow, UNI/UNI-v2 feature bags, MIL training, patient-level cross-validation, and attention heatmaps.

## Expected Inputs

- Slides: `../slideflow_project/data/slides/*.svs`
- Annotations: `../annotations/tcga_crc_msi_annotations.csv`
- Required columns: `slide`, `patient`, `msi_status`, `fold`
- Labels: `MSI-H`, `MSS`

Run `../scripts/make_patient_folds.py` after creating the annotations file if `fold` is not filled.

## 0. Confirm VM, GPU, And Project Setup

Run these first. They confirm that the notebook is using the VM `pathology310` kernel, the NVIDIA GPU is visible to PyTorch, key pathology libraries import, and the project folders are where this notebook expects them.

In [ ]:
from pathlib import Path

def svg_flowchart(title, boxes, filename, width=1200, height=900):
    box_w, box_h = 720, 90
    x = (width - box_w) // 2
    start_y = 90
    gap = 45
    rx = 18

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<defs>',
        '<marker id="arrow" markerWidth="10" markerHeight="10" refX="6" refY="3" orient="auto" markerUnits="strokeWidth">',
        '<path d="M0,0 L0,6 L6,3 z" fill="#333"/>',
        '</marker>',
        '<style>',
        '.title { font: 700 30px Arial, sans-serif; fill: #111; }',
        '.box { fill: #f7f9fc; stroke: #3b4a5a; stroke-width: 2; }',
        '.text { font: 600 22px Arial, sans-serif; fill: #1a1a1a; }',
        '.sub { font: 400 16px Arial, sans-serif; fill: #444; }',
        '.line { stroke: #333; stroke-width: 3; fill: none; marker-end: url(#arrow); }',
        '</style>',
        '</defs>',
        '<rect x="0" y="0" width="100%" height="100%" fill="white"/>',
        f'<text x="{width/2}" y="48" text-anchor="middle" class="title">{title}</text>'
        ,
    ]

    ys = []
    for i, (main, sub) in enumerate(boxes):
        y = start_y + i * (box_h + gap)
        ys.append(y)
        parts.append(
            f'<rect class="box" x="{x}" y="{y}" rx="{rx}" ry="{rx}" width="{box_w}" height="{box_h}"/>'
        )
        parts.append(
            f'<text x="{width/2}" y="{y+36}" text-anchor="middle" class="text">{main}</text>'
        )
        parts.append(
            f'<text x="{width/2}" y="{y+62}" text-anchor="middle" class="sub">{sub}</text>'
        )

    for i in range(len(ys) - 1):
        xmid = width / 2
        y1 = ys[i] + box_h
        y2 = ys[i + 1]
        parts.append(
            f'<line class="line" x1="{xmid}" y1="{y1+6}" x2="{xmid}" y2="{y2-10}"/>'
        )

    parts.append("</svg>")
    Path(filename).write_text("\n".join(parts), encoding="utf-8")


baseline = [
    ("Input Data", "Raw WSIs, slide IDs, patient IDs, MSI labels"),
    ("Data Audit & Split", "QC checks and patient-level train/validation/test split"),
    ("Preprocessing", "Tissue detection, tiling, artifact filtering, normalization"),
    ("Slide Embeddings", "Extract one feature vector per slide"),
    ("Baseline Model", "Train linear probe / logistic regression for MSI-H vs MSS"),
    ("Evaluation & Packaging", "AUROC, calibration, reports, saved model artifacts"),
]

research = [
    ("Input Data", "WSIs, labels, metadata, cohort definitions"),
    ("Experiment Config Layer", "Hydra configs for data, encoder, folds, outputs, hyperparameters"),
    ("Pathology Preprocessing", "TRIDENT or similar for segmentation, tiling, QC, feature extraction"),
    ("MIL Training Layer", "STAMP / Transformer-MIL / Attention-MIL on patch features"),
    ("Optimization & Tracking", "Optuna for tuning, Lightning for training loop, MLflow for tracking"),
    ("Evaluation & Explainability", "Patient-level CV, external validation, attention maps, top tiles"),
    ("Deployment Artifacts", "Export trained weights, configs, metrics, and inference-ready package"),
]

multimodal = [
    ("Inputs", "WSIs + omics + clinical metadata"),
    ("Image Pipeline", "Tiling, feature extraction, slide-level or bag-level embeddings"),
    ("Omics Pipeline", "Normalization, feature selection, pathway or gene-level representation"),
    ("Clinical Pipeline", "Clean metadata, encode variables, handle missing values"),
    ("Fusion Layer", "Early fusion, late fusion, or attention-based multimodal integration"),
    ("MSI Prediction", "Train multimodal classifier for MSI-H vs MSS"),
    ("Biology & Interpretation", "UMAP, clustering, biomarker association, downstream analysis"),
]

svg_flowchart(
    "High-Level MSI Baseline Flowchart",
    baseline,
    "msi_baseline_flowchart.svg",
    height=850,
)

svg_flowchart(
    "High-Level MSI Research Pipeline",
    research,
    "msi_research_flowchart.svg",
    height=980,
)

svg_flowchart(
    "High-Level Multimodal MSI Flowchart",
    multimodal,
    "msi_multimodal_flowchart.svg",
    height=980,
)

print("Generated 3 SVG flowcharts.")

Generated 3 SVG flowcharts.


In [13]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Working directory:', Path.cwd())
print('CONDA_DEFAULT_ENV:', os.environ.get('CONDA_DEFAULT_ENV'))

try:
    import torch
    print('\nTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA version:', torch.version.cuda)
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print('GPU memory GB:', round(props.total_memory / 1024**3, 2))
except Exception as exc:
    print('Torch/GPU check failed:', repr(exc))

if shutil.which('nvidia-smi'):
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True,
        text=True,
    )
    print('\nnvidia-smi:', result.stdout.strip() or result.stderr.strip())
else:
    print('\nnvidia-smi not found on PATH')

Python executable: /opt/miniforge3/envs/pathology310/bin/python
Python version: 3.10.20
Platform: Linux-6.17.0-1010-gcp-x86_64-with-glibc2.42
Working directory: /home/pardeep/pathology310_projects/single_slide_morphology
CONDA_DEFAULT_ENV: None

Torch: 2.7.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA L4
GPU memory GB: 22.03

nvidia-smi: NVIDIA L4, 23034 MiB, 580.126.09


In [15]:
libraries = ['slideflow', 'openslide', 'tiffslide', 'pandas', 'sklearn', 'matplotlib']

for lib in libraries:
    try:
        module = __import__(lib)
        version = getattr(module, '__version__', 'version unknown')
        print(f'{lib}: OK ({version})')
    except Exception as exc:
        print(f'{lib}: FAILED ({exc})')

slideflow: OK (3.0.2)
openslide: OK (1.4.3)
tiffslide: OK (2.5.1)
pandas: OK (2.3.3)
sklearn: OK (1.7.2)
matplotlib: OK (3.10.0)


In [4]:
from pathlib import Path
import importlib.util
import sys

def find_project_dir():
    candidates = [
        Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd() / 'project_1_slideflow_msi_tcga_crc',
        Path.cwd() / 'project_1_slideflow_msi_tcga_crc',
        Path.cwd().parent / 'project_1_slideflow_msi_tcga_crc',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()

PROJECT_DIR = find_project_dir()
SCRIPTS_DIR = PROJECT_DIR / 'scripts'
PIPELINE_SCRIPT = SCRIPTS_DIR / 'run_slideflow_msi_pipeline.py'
FOLD_SCRIPT = SCRIPTS_DIR / 'make_patient_folds.py'
OPEN_DATA_SCRIPT = SCRIPTS_DIR / 'build_open_tcga_crc_dataset.py'

for path in [SCRIPTS_DIR, PROJECT_DIR, Path.cwd()]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

def import_project_module(module_name, module_path):
    if not module_path.exists():
        raise FileNotFoundError(
            f'Missing {module_path}. Copy the whole project_1_slideflow_msi_tcga_crc folder to this VM, not only the notebook.'
        )
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

PROJECT_DIR

def get_pipeline():
    if PIPELINE_SCRIPT.exists():
        return import_project_module('run_slideflow_msi_pipeline', PIPELINE_SCRIPT)

    print(f'Missing helper script: {PIPELINE_SCRIPT}')
    print('Using inline notebook fallback for setup, tiles, features, train, and aggregate.')

    from types import SimpleNamespace

    def require_inputs():
        import pandas as pd
        if not ANNOTATIONS.exists():
            raise FileNotFoundError(f'Missing annotations: {ANNOTATIONS}. Run open_data.main([]) first.')
        if not SLIDES_DIR.exists():
            raise FileNotFoundError(f'Missing slides directory: {SLIDES_DIR}. Download slides with the GDC manifest first.')
        df = pd.read_csv(ANNOTATIONS)
        required = {'slide', 'patient', 'msi_status', 'fold'}
        missing = required.difference(df.columns)
        if missing:
            raise ValueError(f'Missing annotation columns: {sorted(missing)}')
        return df

    def load_or_create_project():
        import slideflow as sf
        SF_ROOT = PROJECT_DIR / 'slideflow_project'
        SF_ROOT.mkdir(parents=True, exist_ok=True)
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        try:
            project = sf.Project(str(SF_ROOT))
        except Exception:
            project = sf.Project(
                str(SF_ROOT),
                name='TCGA_CRC_MSI_Slideflow',
                annotations=str(ANNOTATIONS),
                sources=['tcga_crc_dx'],
                create=True,
            )
        dataset_config = SF_ROOT / 'datasets.json'
        if not dataset_config.exists():
            project.add_source(
                'tcga_crc_dx',
                slides=str(SLIDES_DIR),
                tfrecords=str(SF_ROOT / 'tfrecords'),
                tiles=str(SF_ROOT / 'tiles'),
            )
        return project

    def make_dataset(project, filters=None):
        return project.dataset(
            tile_px=256,
            tile_um=128,
            filters=filters,
            filter_blank='msi_status',
            verification='both',
        )

    def setup():
        df = require_inputs()
        project = load_or_create_project()
        print(project)
        display(df[['patient', 'msi_status']].drop_duplicates()['msi_status'].value_counts())
        display(df[['patient', 'msi_status', 'fold']].drop_duplicates().pivot_table(index='fold', columns='msi_status', values='patient', aggfunc='count', fill_value=0))

    def extract_tiles():
        require_inputs()
        project = load_or_create_project()
        dataset = make_dataset(project)
        dataset.extract_tiles(tile_px=256, tile_um=128, qc='both', normalizer='macenko')

    def build_extractor(sf):
        for name in ['uni_v2', 'uni']:
            try:
                print(f'Trying feature extractor: {name}')
                return sf.build_feature_extractor(name, resize=True, mixed_precision=True)
            except Exception as exc:
                print(f'Could not initialize {name}: {exc}')
        raise RuntimeError('No UNI feature extractor could be initialized.')

    def generate_features():
        require_inputs()
        import slideflow as sf
        project = load_or_create_project()
        dataset = make_dataset(project)
        bags_dir = PROJECT_DIR / 'slideflow_project' / 'bags' / 'uni_v2_256px_128um'
        bags_dir.mkdir(parents=True, exist_ok=True)
        extractor = build_extractor(sf)
        project.generate_feature_bags(extractor, dataset, outdir=str(bags_dir))

    def train():
        require_inputs()
        import slideflow as sf
        project = load_or_create_project()
        dataset = make_dataset(project)
        bags_dir = PROJECT_DIR / 'slideflow_project' / 'bags' / 'uni_v2_256px_128um'
        for model_name in ['transmil', 'attention_mil']:
            try:
                config = sf.mil.mil_config(model_name, lr=1e-4, epochs=20)
                print(f'Using MIL model: {model_name}')
                break
            except Exception as exc:
                print(f'Could not configure {model_name}: {exc}')
        else:
            raise RuntimeError('No MIL model could be configured.')
        for fold in range(1, 6):
            train_ds, val_ds = dataset.split(
                model_type='classification',
                labels='msi_status',
                val_strategy='k-fold-manual',
                val_k_fold_header='fold',
                k_fold_iter=fold,
                splits=str(PROJECT_DIR / 'slideflow_project' / 'splits.json'),
            )
            project.train_mil(
                config=config,
                train_dataset=train_ds,
                val_dataset=val_ds,
                outcomes='msi_status',
                bags=str(bags_dir),
                exp_label=f'msi_{model_name}_fold_{fold}',
                attention_heatmaps=True,
                cmap='magma',
                interpolation=None,
            )

    def aggregate():
        import pandas as pd
        from sklearn.metrics import roc_auc_score, roc_curve
        results_dir = RESULTS_DIR
        results_dir.mkdir(parents=True, exist_ok=True)
        files = list((PROJECT_DIR / 'slideflow_project').glob('**/*pred*.parquet')) + list((PROJECT_DIR / 'slideflow_project').glob('**/*pred*.csv'))
        rows = []
        curves = []
        for path in files:
            df = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
            if 'msi_status' not in df.columns:
                continue
            score_col = next((c for c in ['MSI-H', 'y_pred_MSI-H', 'y_pred1', 'prob_MSI-H', 'prediction'] if c in df.columns), None)
            if score_col is None:
                continue
            y_true = (df['msi_status'] == 'MSI-H').astype(int)
            if y_true.nunique() < 2:
                continue
            y_score = df[score_col]
            fpr, tpr, _ = roc_curve(y_true, y_score)
            fold = len(rows) + 1
            rows.append({'fold': fold, 'n': len(df), 'score_column': score_col, 'auroc': roc_auc_score(y_true, y_score), 'file': str(path)})
            curves.append(pd.DataFrame({'fold': fold, 'fpr': fpr, 'tpr': tpr}))
        if not rows:
            raise FileNotFoundError('No usable prediction files found yet. Run training/evaluation first.')
        table = pd.DataFrame(rows)
        table.to_csv(results_dir / 'cv_auroc_table.csv', index=False)
        pd.concat(curves).to_csv(results_dir / 'roc_curves.csv', index=False)
        display(table)
        print('Mean AUROC:', table['auroc'].mean(), 'SD:', table['auroc'].std(ddof=1))

    return SimpleNamespace(
        setup=setup,
        extract_tiles=extract_tiles,
        generate_features=generate_features,
        train=train,
        aggregate=aggregate,
    )



In [5]:
ANNOTATIONS = PROJECT_DIR / 'annotations' / 'tcga_crc_msi_annotations.csv'
ANNOTATIONS_TEMPLATE = PROJECT_DIR / 'annotations' / 'annotations_template.csv'
SLIDES_DIR = PROJECT_DIR / 'slideflow_project' / 'data' / 'slides'
RESULTS_DIR = PROJECT_DIR / 'slideflow_project' / 'results'

for label, path in {
    'project_dir': PROJECT_DIR,
    'scripts_dir': SCRIPTS_DIR,
    'pipeline_script': PIPELINE_SCRIPT,
    'fold_script': FOLD_SCRIPT,
    'open_data_script': OPEN_DATA_SCRIPT,
    'annotations': ANNOTATIONS,
    'annotations_template': ANNOTATIONS_TEMPLATE,
    'slides_dir': SLIDES_DIR,
    'results_dir': RESULTS_DIR,
}.items():
    print(f'{label}: {path} | exists={path.exists()}')

if SLIDES_DIR.exists():
    svs_files = sorted(SLIDES_DIR.rglob('*.svs'))
    print(f'Found .svs slides: {len(svs_files)}')
    for p in svs_files[:5]:
        print('  ', p.name)
else:
    print('Create or symlink the slides directory before running tile extraction.')

project_dir: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc | exists=True
scripts_dir: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/scripts | exists=True
pipeline_script: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/scripts/run_slideflow_msi_pipeline.py | exists=True
fold_script: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/scripts/make_patient_folds.py | exists=True
open_data_script: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/scripts/build_open_tcga_crc_dataset.py | exists=True
annotations: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/annotations/tcga_crc_msi_annotations.csv | exists=True
annotations_template: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/

In [6]:
import pandas as pd
import shutil

ann = PROJECT_DIR / 'annotations' / 'tcga_crc_msi_annotations.csv'
if ann.exists():
    df = pd.read_csv(ann)
    if df.empty:
        print(f'Annotations file exists but is empty: {ann}')
        print('Fill it with TCGA slide/patient MSI labels before running setup/training.')
    else:
        display(df.head())
        display(df[['patient', 'msi_status']].drop_duplicates()['msi_status'].value_counts())
        if 'fold' in df.columns:
            display(df[['patient', 'msi_status', 'fold']].drop_duplicates().pivot_table(index='fold', columns='msi_status', values='patient', aggfunc='count', fill_value=0))
elif ANNOTATIONS_TEMPLATE.exists():
    ann.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(ANNOTATIONS_TEMPLATE, ann)
    print(f'Created starter annotations file: {ann}')
    print('Fill this CSV with real TCGA slide, patient, msi_status, site, and fold values before running setup/training.')
else:
    ann.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(columns=['slide', 'patient', 'msi_status', 'site', 'fold']).to_csv(ann, index=False)
    print(f'Created empty annotations file: {ann}')
    print('Fill this CSV with real TCGA slide, patient, msi_status, site, and fold values before running setup/training.')

,slide,patient,msi_status,site,fold,project,gdc_file_id,gdc_filename,msi_score_mantis,msi_sensor_score,label_source
0,TCGA-AD-A5EJ-01Z-00-DX1.FA56CEAF-8B70-45EF-A2C...,TCGA-AD-A5EJ,MSI-H,AD,1,TCGA-COAD,5fb8d491-f1b7-4284-b982-a965b12edcd3,TCGA-AD-A5EJ-01Z-00-DX1.FA56CEAF-8B70-45EF-A2C...,1.0934,22.40,cBioPortal PanCancer Atlas; MANTIS/MSIsensor t...
1,TCGA-CM-6674-01Z-00-DX1.4a08b16a-788e-43dc-85d...,TCGA-CM-6674,MSI-H,CM,1,TCGA-COAD,645ac325-9832-46ab-9832-9652bf6add98,TCGA-CM-6674-01Z-00-DX1.4a08b16a-788e-43dc-85d...,0.7398,21.08,cBioPortal PanCancer Atlas; MANTIS/MSIsensor t...
2,TCGA-NH-A5IV-01Z-00-DX1.1A6F9F12-C00E-47F2-940...,TCGA-NH-A5IV,MSI-H,NH,1,TCGA-COAD,a8903dd1-b3ec-4b10-89f3-142fab914157,TCGA-NH-A5IV-01Z-00-DX1.1A6F9F12-C00E-47F2-940...,1.2012,41.02,cBioPortal PanCancer Atlas; MANTIS/MSIsensor t...
3,TCGA-QG-A5Z2-01Z-00-DX1.51896C31-235E-48EF-90F...,TCGA-QG-A5Z2,MSI-H,QG,1,TCGA-COAD,f4e7f126-b2e2-4e0e-b7cc-4bf4df4656f9,TCGA-QG-A5Z2-01Z-00-DX1.51896C31-235E-48EF-90F...,0.8600,19.29,cBioPortal PanCancer Atlas; MANTIS/MSIsensor t...
4,TCGA-AA-3675-01Z-00-DX1.9afbbb26-2574-46af-815...,TCGA-AA-3675,MSS,AA,1,TCGA-COAD,b7c23d21-c9b8-4d40-b32d-01a7081e5189,TCGA-AA-3675-01Z-00-DX1.9afbbb26-2574-46af-815...,0.2618,0.31,cBioPortal PanCancer Atlas; MANTIS/MSIsensor t...


msi_status
MSS      40
MSI-H    20
Name: count, dtype: int64

msi_status,MSI-H,MSS
fold,,
1,4,8
2,4,8
3,4,8
4,4,8
5,4,8


## 1. Build Open TCGA-CRC Cohort

This uses open sources: cBioPortal PanCancer Atlas MSI scores for labels and GDC diagnostic slide metadata for `.svs` files. It writes the annotations CSV and a GDC manifest for the selected slides.

In [7]:
from types import SimpleNamespace

if OPEN_DATA_SCRIPT.exists():
    open_data = import_project_module('build_open_tcga_crc_dataset', OPEN_DATA_SCRIPT)
else:
    print(f'Missing helper script: {OPEN_DATA_SCRIPT}')
    print('Using inline notebook fallback to build the open TCGA-CRC cohort.')

    def build_open_tcga_crc_dataset(msi_h=20, mss=40, seed=310):
        import random
        from collections import Counter, defaultdict
        import pandas as pd
        import requests

        cbio_study = 'coadread_tcga_pan_can_atlas_2018'
        cbio_url = f'https://www.cbioportal.org/api/studies/{cbio_study}/clinical-data'
        gdc_url = 'https://api.gdc.cancer.gov/files'
        annotations_dir = PROJECT_DIR / 'annotations'
        annotations_dir.mkdir(parents=True, exist_ok=True)
        SLIDES_DIR.mkdir(parents=True, exist_ok=True)
        random.seed(seed)

        print('Fetching MSI scores from cBioPortal...')
        clinical = requests.get(
            cbio_url,
            params={
                'clinicalDataType': 'SAMPLE',
                'projection': 'DETAILED',
                'pageSize': 100000,
            },
            timeout=120,
        )
        clinical.raise_for_status()
        clinical_rows = clinical.json()

        by_sample = defaultdict(dict)
        for row in clinical_rows:
            attr = row.get('clinicalAttributeId')
            if attr not in {'MSI_SCORE_MANTIS', 'MSI_SENSOR_SCORE'}:
                continue
            sample_id = row['sampleId']
            by_sample[sample_id]['sample_id'] = sample_id
            by_sample[sample_id]['patient'] = row['patientId']
            by_sample[sample_id][attr] = row.get('value')

        def to_float(value):
            try:
                if value in {None, '', 'NA', 'N/A'}:
                    return None
                return float(value)
            except Exception:
                return None

        def classify(mantis, sensor):
            if mantis is not None:
                if mantis > 0.6:
                    return 'MSI-H'
                if mantis < 0.4:
                    return 'MSS'
                return None
            if sensor is not None:
                if sensor > 10:
                    return 'MSI-H'
                if sensor < 4:
                    return 'MSS'
            return None

        msi_rows = []
        for value in by_sample.values():
            mantis = to_float(value.get('MSI_SCORE_MANTIS'))
            sensor = to_float(value.get('MSI_SENSOR_SCORE'))
            status = classify(mantis, sensor)
            if status is None:
                continue
            msi_rows.append({
                'sample_id': value['sample_id'],
                'patient': value['patient'],
                'msi_score_mantis': mantis,
                'msi_sensor_score': sensor,
                'msi_status': status,
                'label_source': 'cBioPortal PanCancer Atlas; MANTIS/MSIsensor thresholds',
            })

        print('Fetching diagnostic .svs slide metadata from GDC...')
        filters = {
            'op': 'and',
            'content': [
                {'op': 'in', 'content': {'field': 'cases.project.project_id', 'value': ['TCGA-COAD', 'TCGA-READ']}},
                {'op': 'in', 'content': {'field': 'data_type', 'value': ['Slide Image']}},
                {'op': 'in', 'content': {'field': 'experimental_strategy', 'value': ['Diagnostic Slide']}},
                {'op': 'in', 'content': {'field': 'data_format', 'value': ['SVS']}},
            ],
        }
        fields = ','.join([
            'file_id', 'file_name', 'md5sum', 'file_size', 'state',
            'cases.submitter_id', 'cases.project.project_id', 'cases.samples.sample_type',
            'data_type', 'data_format', 'experimental_strategy',
        ])
        response = requests.post(gdc_url, json={'filters': filters, 'fields': fields, 'format': 'JSON', 'size': 5000}, timeout=120)
        response.raise_for_status()
        hits = response.json()['data']['hits']

        slide_by_patient = {}
        grouped = defaultdict(list)
        for hit in hits:
            case = (hit.get('cases') or [{}])[0]
            patient = case.get('submitter_id')
            if not patient:
                continue
            samples = case.get('samples') or []
            sample_type = samples[0].get('sample_type', '') if samples else ''
            filename = hit['file_name']
            grouped[patient].append({
                'id': hit.get('file_id') or hit.get('id'),
                'filename': filename,
                'md5': hit.get('md5sum', ''),
                'size': hit.get('file_size', ''),
                'state': hit.get('state', 'released'),
                'slide': filename[:-4] if filename.lower().endswith('.svs') else filename,
                'patient': patient,
                'project': case.get('project', {}).get('project_id', ''),
                'sample_type': sample_type,
                'site': patient.split('-')[1] if '-' in patient else '',
            })

        for patient, rows in grouped.items():
            rows = sorted(rows, key=lambda r: ('DX1' not in r['filename'], 'Primary Tumor' not in r.get('sample_type', ''), r['filename']))
            slide_by_patient[patient] = rows[0]

        joined = []
        for label in msi_rows:
            slide = slide_by_patient.get(label['patient'])
            if slide:
                joined.append({**slide, **label})

        by_status = defaultdict(list)
        for row in joined:
            by_status[row['msi_status']].append(row)
        for rows in by_status.values():
            random.shuffle(rows)

        selected = by_status['MSI-H'][:msi_h] + by_status['MSS'][:mss]
        random.shuffle(selected)
        by_label = defaultdict(list)
        for row in selected:
            by_label[row['msi_status']].append(row)
        for rows in by_label.values():
            for idx, row in enumerate(rows):
                row['fold'] = (idx % 5) + 1
        selected = sorted(selected, key=lambda r: (r['fold'], r['msi_status'], r['patient']))

        annotations = pd.DataFrame([{
            'slide': r['slide'],
            'patient': r['patient'],
            'msi_status': r['msi_status'],
            'site': r['site'],
            'fold': r['fold'],
            'project': r['project'],
            'gdc_file_id': r['id'],
            'gdc_filename': r['filename'],
            'msi_score_mantis': r['msi_score_mantis'],
            'msi_sensor_score': r['msi_sensor_score'],
            'label_source': r['label_source'],
        } for r in selected])
        manifest = pd.DataFrame([{
            'id': r['id'],
            'filename': r['filename'],
            'md5': r['md5'],
            'size': r['size'],
            'state': r['state'],
        } for r in selected])

        annotations.to_csv(annotations_dir / 'tcga_crc_msi_annotations.csv', index=False)
        manifest.to_csv(annotations_dir / 'gdc_manifest_tcga_crc_msi.tsv', sep='	', index=False)
        pd.DataFrame(selected).to_csv(annotations_dir / 'selected_open_tcga_crc_slides.csv', index=False)
        pd.DataFrame(msi_rows).to_csv(annotations_dir / 'cbioportal_msi_scores.csv', index=False)

        print('Selected:', dict(Counter(annotations['msi_status'])))
        print('Fold table:')
        display(annotations.pivot_table(index='fold', columns='msi_status', values='patient', aggfunc='count', fill_value=0))
        print('Wrote:', annotations_dir / 'tcga_crc_msi_annotations.csv')
        print('Wrote:', annotations_dir / 'gdc_manifest_tcga_crc_msi.tsv')

    open_data = SimpleNamespace(main=lambda argv=None: build_open_tcga_crc_dataset())

# Uncomment to rebuild the open cohort files.
# open_data.main([])


In [8]:
manifest = PROJECT_DIR / 'annotations' / 'gdc_manifest_tcga_crc_msi.tsv'
print('Manifest:', manifest, '| exists=', manifest.exists())
print('Slides destination:', SLIDES_DIR)
print('\nRun on the VM shell after creating the manifest:')
print(f'gdc-client download -m {manifest} -d {SLIDES_DIR}')

Manifest: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/annotations/gdc_manifest_tcga_crc_msi.tsv | exists= True
Slides destination: /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/slideflow_project/data/slides

Run on the VM shell after creating the manifest:
gdc-client download -m /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/annotations/gdc_manifest_tcga_crc_msi.tsv -d /home/pardeep/pathology310_projects/single_slide_morphology/project_1_slideflow_msi_tcga_crc/slideflow_project/data/slides


## Setup And Tile Extraction

This creates/loads the Slideflow project, verifies annotations/slides, and extracts 256 px / 128 um tiles with both QC and Macenko normalization.

In [9]:
pipeline = get_pipeline()

# pipeline.setup()
# Uncomment for the heavy stage after slides are present.
# pipeline.extract_tiles()

## Feature Bags

Uses `uni_v2` first, then falls back to `uni` if the installed Slideflow package does not expose `uni_v2`. Make sure Hugging Face access is approved and authenticated for UNI-family models.

In [10]:
pipeline = get_pipeline()

# Uncomment for the heavy stage after tiles are extracted.
# pipeline.generate_features()

## 5-Fold Patient-Level MIL Training

Uses manual patient folds from the `fold` column. The script tries `transmil` first and falls back to `attention_mil` if TransMIL is not registered in this Slideflow install.

In [11]:
pipeline = get_pipeline()

# Uncomment after feature bags exist.
# pipeline.train()

## Aggregate AUROC And Curves

After training, this searches the Slideflow project for prediction tables, infers the MSI-H score column, and writes `cv_auroc_table.csv`, `roc_curves.csv`, and `cv_summary.json` under `slideflow_project/results`.

In [12]:
pipeline = get_pipeline()

# Uncomment after training/evaluation outputs exist.
# pipeline.aggregate()

## Review Attention Heatmaps

Slideflow saves attention arrays and heatmaps in the MIL/evaluation output folders when `attention_heatmaps=True`. Pick the top MSI-H predicted validation slides and review whether high-attention regions correspond to tumor-infiltrating lymphocytes, mucinous morphology, poor differentiation, or artifacts.